In [0]:
from pyspark.sql import functions as Ffrom delta.tables import DeltaTable

In [0]:
%run /Workspace/consolidated_pipeline/1_setup/utilities%run /Workspace/consolidated_pipeline/1_setup/s3_ingestion

In [0]:
print(bronze_schema, silver_schema, gold_schema)

In [0]:
dbutils.widgets.text("catalog", "fmcg", "Catalog")dbutils.widgets.text("data_source", "orders", "Data Source")catalog = dbutils.widgets.get("catalog")data_source = dbutils.widgets.get("data_source")base_path = f"/Volumes/{catalog}/bronze/s3/{data_source}"landing_path = f"{base_path}/landing/"processed_path = f"{base_path}/processed/"stage_s3_folder(    data_source,    catalog,    f"{data_source}/landing",    f"{data_source}/processed",    skip_existing=True,)print("Base Path: ", base_path)print("Landing Path: ", landing_path)print("Processed Path: ", processed_path)bronze_table = f"{catalog}.{bronze_schema}.{data_source}"silver_table = f"{catalog}.{silver_schema}.{data_source}"gold_table = f"{catalog}.{gold_schema}.sb_fact_{data_source}"

In [0]:
df = spark.read.options(header=True, inferSchema=True).csv(f"{landing_path}/*.csv").withColumn("read_timestamp", F.current_timestamp()).select("*", "_metadata.file_name", "_metadata.file_size")print("Total Rows: ", df.count())df.show(5)

In [0]:
df.write.format("delta").option("delta.enableChangeDataFeed", "true").mode("append").saveAsTable(bronze_table)